# 05 — Extract Backbone & Neck Features

**Goal:** Extract intermediate feature maps from the fine-tuned YOLO26 for future reuse.

This notebook:
1. Loads the BDD100K-trained YOLO26 weights
2. Inspects internal modules
3. Registers forward hooks on backbone and neck layers
4. Runs inference and captures feature maps
5. Prints feature tensor shapes
6. Visualises selected feature maps
7. Makes this reusable as a feature extractor for lane marking work

## 1 — Setup

In [ ]:
!pip install -q ultralytics>=8.4.0 opencv-python matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
import cv2
from collections import OrderedDict

ECOCAR_ROOT = "/content/drive/MyDrive/EcoCAR"
WEIGHTS_PATH = os.path.join(ECOCAR_ROOT, "weights", "bdd100k_yolo26s_best.pt")
YOLO_DATASET_DIR = os.path.join(ECOCAR_ROOT, "datasets", "bdd100k_yolo")
OUTPUT_DIR = os.path.join(ECOCAR_ROOT, "outputs", "05_features")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# GPU check
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
print(f"Weights: {WEIGHTS_PATH} (exists: {os.path.isfile(WEIGHTS_PATH)})")

## 2 — Load Model

In [ ]:
from ultralytics import YOLO

model = YOLO(WEIGHTS_PATH)
print(f"✅ Model loaded: {len(model.names)} classes")

## 3 — Inspect Model Architecture

In [ ]:
# List all top-level layers with indices
inner_model = model.model  # The inner nn.Module

try:
    layers = list(inner_model.model)
except (TypeError, AttributeError):
    layers = list(inner_model.children())

print(f"\n{'Idx':>4} {'Module':<35} {'Params':>10}")
print("─" * 55)
for idx, layer in enumerate(layers):
    params = sum(p.numel() for p in layer.parameters())
    cls_name = layer.__class__.__name__
    print(f"  {idx:>2}  {cls_name:<35} {params:>10,}")

print(f"\nTotal layers: {len(layers)}")

## 4 — Define Feature Extractor

We use forward hooks to capture intermediate feature maps without modifying the model.

In [ ]:
class FeatureExtractor:
    """
    Register forward hooks on YOLO model layers to capture feature maps.
    
    Usage:
        extractor = FeatureExtractor(model.model)
        extractor.register_hooks(layer_indices=[2, 4, 6, 8])
        results = model("image.jpg")  # triggers hooks
        features = extractor.get_features()
        extractor.remove_hooks()
    """
    
    def __init__(self, model):
        self.model = model
        self._features = OrderedDict()
        self._hooks = []
    
    def _make_hook(self, name):
        def hook_fn(module, input, output):
            if isinstance(output, torch.Tensor):
                self._features[name] = output.detach()
            elif isinstance(output, (list, tuple)):
                for item in output:
                    if isinstance(item, torch.Tensor):
                        self._features[name] = item.detach()
                        break
        return hook_fn
    
    def register_hooks(self, layer_indices):
        self.remove_hooks()
        self._features.clear()
        
        try:
            layers = list(self.model.model)
        except (TypeError, AttributeError):
            layers = list(self.model.children())
        
        registered = []
        for idx in layer_indices:
            if idx < 0 or idx >= len(layers):
                print(f"⚠ Index {idx} out of range")
                continue
            name = f"layer_{idx}_{layers[idx].__class__.__name__}"
            h = layers[idx].register_forward_hook(self._make_hook(name))
            self._hooks.append(h)
            registered.append(name)
        
        print(f"✅ Registered {len(registered)} hooks")
        return registered
    
    def get_features(self):
        return self._features
    
    def get_feature_shapes(self):
        return {k: tuple(v.shape) for k, v in self._features.items()}
    
    def remove_hooks(self):
        for h in self._hooks:
            h.remove()
        self._hooks.clear()
    
    def clear_features(self):
        self._features.clear()

## 5 — Register Hooks & Extract Features

In [ ]:
# Select layers to hook
# Typical YOLO architecture:
#   Layers 0-9:  Backbone (feature extraction)
#   Layers 10+:  Neck (FPN/PAN feature fusion)
#   Last layer:  Detection head
# Adjust these based on the architecture printout above

BACKBONE_LAYERS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
NECK_LAYERS = [10, 12, 15, 18, 21]  # Key neck fusion points

ALL_HOOK_LAYERS = BACKBONE_LAYERS + NECK_LAYERS

extractor = FeatureExtractor(model.model)
extractor.register_hooks(ALL_HOOK_LAYERS)

In [ ]:
# Select a test image
val_images_dir = os.path.join(YOLO_DATASET_DIR, "images", "val")

if os.path.isdir(val_images_dir):
    import random
    img_files = [f for f in os.listdir(val_images_dir) if f.endswith(('.jpg', '.png'))]
    test_image = os.path.join(val_images_dir, random.choice(img_files))
else:
    # Fallback
    import urllib.request
    test_image = "test_driving.jpg"
    if not os.path.exists(test_image):
        urllib.request.urlretrieve("https://ultralytics.com/images/bus.jpg", test_image)

print(f"Test image: {test_image}")

# Run inference (this triggers the hooks)
results = model(test_image, verbose=False)

print(f"\n✅ Inference complete. {len(results[0].boxes)} detections.")

## 6 — Print Feature Tensor Shapes

In [ ]:
features = extractor.get_features()

print(f"\n{'Layer':<45} {'Shape':<25} {'Size (MB)':>10}")
print("─" * 82)

total_mb = 0
for name, feat in features.items():
    shape_str = str(tuple(feat.shape))
    size_mb = feat.nelement() * feat.element_size() / (1024**2)
    total_mb += size_mb
    
    # Mark backbone vs neck
    layer_idx = int(name.split('_')[1])
    region = "[BACKBONE]" if layer_idx <= 9 else "[NECK]    "
    
    print(f"  {region} {name:<35} {shape_str:<25} {size_mb:>8.2f} MB")

print(f"\n  Total feature memory: {total_mb:.2f} MB")
print(f"  Features captured: {len(features)}")

## 7 — Visualise Feature Maps

In [ ]:
# Show the original image first
img = cv2.imread(test_image)
plt.figure(figsize=(12, 6))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title(f"Input: {os.path.basename(test_image)}")
plt.axis('off')
plt.show()

In [ ]:
# Visualise a selection of feature maps
# Pick a few interesting layers (early, mid, late backbone + neck)
vis_layers = list(features.keys())
# Select ~4 layers spread across the network
if len(vis_layers) > 4:
    step = len(vis_layers) // 4
    vis_layers = [vis_layers[i] for i in range(0, len(vis_layers), step)][:4]

for layer_name in vis_layers:
    feat = features[layer_name]
    if feat.dim() < 4:
        print(f"  Skipping {layer_name} (shape {tuple(feat.shape)})")
        continue
    
    feat_2d = feat[0].cpu()  # (C, H, W)
    n_show = min(8, feat_2d.shape[0])
    
    fig, axes = plt.subplots(1, n_show, figsize=(16, 3))
    fig.suptitle(f"{layer_name} — shape: {tuple(feat.shape)}", fontsize=10)
    
    for ch in range(n_show):
        axes[ch].imshow(feat_2d[ch].numpy(), cmap='viridis')
        axes[ch].set_title(f"ch {ch}", fontsize=8)
        axes[ch].axis('off')
    
    plt.tight_layout()
    
    # Save
    save_path = os.path.join(OUTPUT_DIR, f"featuremap_{layer_name}.png")
    plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.show()
    print(f"  💾 {save_path}")

## 8 — Mean Feature Activation Heatmaps

In [ ]:
# Average across channels for a heatmap view
fig, axes = plt.subplots(1, len(vis_layers), figsize=(5 * len(vis_layers), 4))
if len(vis_layers) == 1:
    axes = [axes]

for i, layer_name in enumerate(vis_layers):
    feat = features[layer_name]
    if feat.dim() < 4:
        continue
    
    mean_activation = feat[0].cpu().mean(dim=0).numpy()  # (H, W)
    axes[i].imshow(mean_activation, cmap='hot')
    axes[i].set_title(f"Layer {layer_name.split('_')[1]}\n{feat.shape[2]}×{feat.shape[3]}", fontsize=9)
    axes[i].axis('off')

plt.suptitle('Mean Feature Activation Heatmaps (averaged across channels)', fontsize=11)
plt.tight_layout()

save_path = os.path.join(OUTPUT_DIR, "mean_activation_heatmaps.png")
plt.savefig(save_path, dpi=120, bbox_inches='tight')
plt.show()
print(f"💾 {save_path}")

## 9 — Reusable Feature Extraction Function

In [ ]:
def extract_features_from_image(model, image_path, layer_indices=None):
    """
    One-shot feature extraction helper.
    
    Args:
        model:         Ultralytics YOLO model
        image_path:    Path to image
        layer_indices: List of layer indices (default: backbone)
    
    Returns:
        OrderedDict of {name: tensor}
    """
    if layer_indices is None:
        layer_indices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]  # backbone
    
    ext = FeatureExtractor(model.model)
    ext.register_hooks(layer_indices)
    _ = model(image_path, verbose=False)
    feats = ext.get_features()
    ext.remove_hooks()
    return feats

# Demo
demo_features = extract_features_from_image(model, test_image, layer_indices=[4, 6, 9])
print("\nReusable extraction demo:")
for name, feat in demo_features.items():
    print(f"  {name}: {tuple(feat.shape)}")

## 10 — Cleanup & Summary

In [ ]:
# Remove hooks
extractor.remove_hooks()

print("\n" + "="*60)
print(" FEATURE EXTRACTION SUMMARY")
print("="*60)
print(f" Model:             {WEIGHTS_PATH}")
print(f" Backbone layers:   {BACKBONE_LAYERS}")
print(f" Neck layers:       {NECK_LAYERS}")
print(f" Features captured: {len(features)}")
print(f" Total memory:      {total_mb:.2f} MB")
print(f" Outputs saved to:  {OUTPUT_DIR}")
print("="*60)
print("\n📌 For future lane marking work:")
print("   • Use backbone features (layers 4-9) as input to a lane decoder")
print("   • Use neck features for multi-scale predictions")
print("   • The FeatureExtractor class above is ready for reuse")
print("\n🎯 Next: notebook 06 for detailed model inspection")